In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('D:/Important Files/Machine Learning/Datasets/play_tennis.csv')
df.drop(columns=['day'], inplace=True)

In [3]:
df.head()

,outlook,temp,humidity,wind,play
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rain,Mild,High,Weak,Yes
4,Rain,Cool,Normal,Weak,Yes


In [4]:
df['play'].value_counts()

play
Yes    9
No     5
Name: count, dtype: int64

## Naive Bayes Classifier — Manual Calculation:

This notebook works through Naive Bayes classification manually, step by step, on the classic "Play Tennis" dataset. Rather than using a library implementation, each conditional probability is computed by hand from frequency tables, to build intuition for how Bayes' Theorem combines them into a final classification decision.

**Bayes' Theorem for classification:**

$$P(y \mid X) = \frac{P(X \mid y) \cdot P(y)}{P(X)}$$

The **"naive"** part comes from assuming all features are conditionally independent given the class — so instead of computing one complex joint probability $P(X \mid y)$, it's approximated as the product of each individual feature's conditional probability:

$$P(X \mid y) \approx P(x_1 \mid y) \cdot P(x_2 \mid y) \cdot ... \cdot P(x_n \mid y)$$

In [5]:
prob_yes = 9/14
prob_no = 5/14

In [6]:
print(f'Probability of playing tennis: {prob_yes:.2f}')
print(f'Probability of not playing tennis: {prob_no:.2f}')

Probability of playing tennis: 0.64
Probability of not playing tennis: 0.36


### Step 2: Conditional Probabilities per Feature:

For each feature, compute $P(\text{feature value} \mid \text{class})$ using frequency counts from the data.

In [7]:
pd.crosstab(df['outlook'], df['play'])

play,No,Yes
outlook,,
Overcast,0,4
Rain,2,3
Sunny,3,2


In [8]:
prob_overcast_no = 0
prob_rain_no = 2/5
prob_sunny_no = 3/5

prob_overcast_yes = 4/9
prob_rain_yes = 2/9
prob_sunny_yes = 3/9

In [9]:
pd.crosstab(df['temp'], df['play'])

play,No,Yes
temp,,
Cool,1,3
Hot,2,2
Mild,2,4


In [10]:
prob_cool_no = 1/5
prob_hot_no = 2/5
prob_mild_no = 2/5

prob_cool_yes = 3/9
prob_hot_yes = 2/9
prob_mild_yes = 4/9

In [11]:
pd.crosstab(df['humidity'], df['play'])

play,No,Yes
humidity,,
High,4,3
Normal,1,6


In [12]:
prob_high_no = 4/5
prob_normal_no = 1/5

prob_high_yes = 3/9
prob_normal_yes = 6/9

In [13]:
pd.crosstab(df['wind'], df['play'])

play,No,Yes
wind,,
Strong,3,3
Weak,2,6


In [14]:
prob_strong_no = 3/5
prob_weak_no = 2/5

prob_strong_yes = 3/9
prob_weak_yes = 6/9

### Step 3: Applying Bayes' Theorem for a New Sample:

Given a new sample's feature values, multiply the relevant conditional probabilities together with the class prior, for both classes. Whichever result is larger is the predicted class.

**Note:** the results below are *unnormalized* — proportional scores useful for comparison, not true probabilities (they don't sum to 1). Normalizing by dividing by their sum is shown at the end.

In [15]:
# New sample: Outlook=Sunny, Temp=Cool, Humidity=High, Wind=Strong

prob_yes_given_X = prob_sunny_yes * prob_cool_yes * prob_high_yes * prob_strong_yes * prob_yes
print(f'Unnormalized probability of playing tennis given the conditions: {prob_yes_given_X:.4f}')

Unnormalized probability of playing tennis given the conditions: 0.0079


In [16]:
# New sample: Outlook=Sunny, Temp=Cool, Humidity=High, Wind=Strong

prob_no_given_X = prob_sunny_no * prob_cool_no * prob_high_no * prob_strong_no * prob_no
print(f'Unnormalized probability of not playing tennis given the conditions: {prob_no_given_X:.4f}')

Unnormalized probability of not playing tennis given the conditions: 0.0206


In [17]:
total = prob_yes_given_X + prob_no_given_X

normalized_yes = prob_yes_given_X / total
normalized_no = prob_no_given_X / total

print(f'Normalized probability of playing tennis: {normalized_yes:.4f}')
print(f'Normalized probability of not playing tennis: {normalized_no:.4f}')

prediction = 'Yes' if prob_yes_given_X > prob_no_given_X else 'No'
print(f'Predicted class: {prediction}')

Normalized probability of playing tennis: 0.2784
Normalized probability of not playing tennis: 0.7216
Predicted class: No


### Result:

Since $P(\text{Yes} \mid X) > P(\text{No} \mid X)$ for the first sample, Naive Bayes predicts the class as **Yes** (play tennis) for that combination of conditions.